# Feature Set: manual_extended

**관점**: 요약 통계 (구매력·다양성·행동·집중도 + 엔트로피·충성도·재구매율·변동성)

이 노트북은 **하나의 feature set** 에 대해 5-fold 외부 CV로 AutoGluon(블랙박스 모델)을 학습하고, CV 지표(AUC/LogLoss/Brier)를 확인한 뒤 **OOF 예측**과 **test 예측**을 저장합니다.

- `oof_{setname}.csv` : train OOF (메타모델 학습 입력)
- `oof_test_{setname}.csv` : test 예측 평균 (메타모델 추론 입력)

> 모든 set 노트북은 동일한 `folds.csv` split 을 공유합니다 (스태킹 정합성).

## 0. 설치 & 라이브러리

In [4]:
import sys
!{sys.executable} -m pip install ipython-autotime
!{sys.executable} -m pip install autogluon.tabular gensim

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [6]:
%load_ext autotime

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from autogluon.tabular import TabularPredictor
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('autogluon').setLevel(logging.ERROR)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# ---------------- 전역 설정 ----------------
SEED        = 42
N_SPLITS    = 5
TIME_LIMIT  = 3600          # 1800
PRESETS     = 'best_quality'
EVAL_METRIC = 'roc_auc'    # AutoGluon 내부 모델 선택 기준

DATA_DIR = '../../data/raw'
OOF_DIR  = '../../outputs/oof'
os.makedirs(OOF_DIR, exist_ok=True)

time: 1.88 s (started: 2026-06-07 17:36:08 +09:00)


## 1. 데이터 로드

In [8]:
# 거래 단위 원본 데이터 로드
X_train_trans = pd.read_csv(f'{DATA_DIR}/train_transactions.csv', encoding='cp949')
X_test_trans  = pd.read_csv(f'{DATA_DIR}/test_transactions.csv',  encoding='cp949')
y_train       = pd.read_csv(f'{DATA_DIR}/y_train.csv',            encoding='cp949')

print('train trans:', X_train_trans.shape, '| test trans:', X_test_trans.shape)
print('y_train:', y_train.shape)

train trans: (1036653, 16) | test trans: (689777, 16)
y_train: (30000, 2)
time: 3.14 s (started: 2026-06-07 17:36:13 +09:00)


## 2. Fold split 로드/생성 (공유)

In [11]:
# ===== Fold split: 모든 feature set / 노트북이 공유 =====
# 첫 실행에서 folds.csv 가 없으면 생성, 있으면 로드 -> 7개 set 모두 동일 split 사용 보장
FOLDS_PATH = f'{OOF_DIR}/folds.csv'

if os.path.exists(FOLDS_PATH):
    folds_df = pd.read_csv(FOLDS_PATH)
    print('기존 folds.csv 로드:', folds_df.shape)
else:
    base = y_train[['custid', 'gender']].sort_values('custid').reset_index(drop=True)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    base['fold'] = -1
    for f, (_, val_idx) in enumerate(skf.split(base['custid'], base['gender'])):
        base.loc[val_idx, 'fold'] = f
    folds_df = base[['custid', 'fold']]
    folds_df.to_csv(FOLDS_PATH, index=False)
    print('folds.csv 신규 생성:', folds_df.shape)

print(folds_df['fold'].value_counts().sort_index())

기존 folds.csv 로드: (30000, 2)
fold
0    6000
1    6000
2    6000
3    6000
4    6000
Name: count, dtype: int64
time: 15 ms (started: 2026-06-07 17:36:17 +09:00)


## 3. Feature 생성  ← 팀원이 통째로 교체하는 영역
`features_train`, `features_test` 두 변수를 `custid` + feature 컬럼 형태로 만들면 됩니다.

In [14]:
SETNAME = 'manual_extended'

time: 0 ns (started: 2026-06-07 17:36:18 +09:00)


In [16]:
# ===== FEATURE 생성 (수정/교체 가능): Manual_Extended =====
# 관점: 요약 통계 (구매력/다양성/행동/집중도) + 엔트로피·충성도·재구매율·변동성 확장
def make_features(df):
    df = df.copy()

    # ── 공통 파생 컬럼 ──
    df['sales_datetime'] = pd.to_datetime(df['sales_datetime'])
    df['sales_day']      = df['sales_datetime'].dt.normalize()
    df['month']          = df['sales_datetime'].dt.month
    df['weekday']        = df['sales_datetime'].dt.weekday
    df['hour']           = df['sales_datetime'].dt.hour
    df['quarter']        = df['sales_datetime'].dt.quarter
    df['is_weekend']     = df['weekday'].isin([5, 6]).astype(int)
    df['is_monday']      = (df['weekday'] == 0).astype(int)
    df['is_night_old']   = (df['hour'] >= 18).astype(int)
    df['is_night_new']   = df['hour'].between(20, 23).astype(int)
    df['is_early']       = df['hour'].between(6, 9).astype(int)
    df['is_refund']      = (df['net_amt'] < 0).astype(int)
    df['is_installment'] = (df['inst_mon'] > 1).astype(int)
    df['is_high_price']  = (df['net_amt'] > 200_000).astype(int)
    df['is_low_price']   = ((df['net_amt'] > 0) & (df['net_amt'] < 30_000)).astype(int)
    df['is_cosmetic']    = (df['buyer_nm'] == '화장품').astype(int)
    df['is_luxury']      = (df['buyer_nm'] == '수입명품').astype(int)
    season_map = {3:'봄',4:'봄',5:'봄', 6:'여름',7:'여름',8:'여름',
                  9:'가을',10:'가을',11:'가을', 12:'겨울',1:'겨울',2:'겨울'}
    df['season'] = df['month'].map(season_map)

    g = df.groupby('custid')
    max_date = df['sales_datetime'].max()

    # ① 기본/다양성 (13개)
    base = g.agg(
        total_txn_cnt  = ('goodcd',   'size'),
        unique_goods   = ('goodcd',   'nunique'),
        unique_brand   = ('brd_nm',   'nunique'),
        unique_corner  = ('corner_nm','nunique'),
        unique_pc      = ('pc_nm',    'nunique'),
        unique_part    = ('part_nm',  'nunique'),
        unique_team    = ('team_nm',  'nunique'),
        unique_store   = ('str_nm',   'nunique'),
        visit_days     = ('sales_day','nunique'),
    )
    base['goods_diversity'] = base['unique_goods']  / base['total_txn_cnt']
    base['brand_diversity'] = base['unique_brand']  / base['total_txn_cnt']
    base['txn_per_visit']   = base['total_txn_cnt'] / base['visit_days']
    base['repurchase_rate'] = 1 - base['unique_goods'] / base['total_txn_cnt']

    # ② 금액 (12개)
    amt = g.agg(
        total_net  = ('net_amt','sum'),    mean_net   = ('net_amt','mean'),
        max_net    = ('net_amt','max'),    min_net    = ('net_amt','min'),
        std_net    = ('net_amt','std'),    median_net = ('net_amt','median'),
        total_tot  = ('tot_amt','sum'),    total_dis  = ('dis_amt','sum'),
    )
    amt['discount_rate'] = amt['total_dis'] / (amt['total_tot'].abs() + 1)
    amt['amt_per_visit'] = amt['total_net'] / base['visit_days']
    amt['net_cv']        = amt['std_net']   / (amt['mean_net'].abs() + 1)
    amt['max_to_mean']   = amt['max_net']   / (amt['mean_net'].abs() + 1)

    # ③ 행동패턴 (6개)
    behavior = g.agg(
        refund_ratio      = ('is_refund',      'mean'),
        import_ratio      = ('import_flg',     'mean'),
        installment_ratio = ('is_installment', 'mean'),
        mean_inst_mon     = ('inst_mon',        'mean'),
        weekend_ratio     = ('is_weekend',     'mean'),
        night_ratio       = ('is_night_old',   'mean'),
    )

    # ④ 엔트로피/충성도 (5개)
    def entropy(s):
        p = s.value_counts(normalize=True)
        return -(p * np.log(p + 1e-12)).sum()
    def top_ratio(s):
        return s.value_counts(normalize=True).iloc[0]

    ent = g.agg(
        brand_entropy   = ('brd_nm',  entropy),
        team_entropy    = ('team_nm', entropy),
        store_loyalty   = ('str_nm',  top_ratio),
        top_brand_ratio = ('brd_nm',  top_ratio),
        top_team_ratio  = ('team_nm', top_ratio),
    )

    # ⑤⑥⑦ ratio 계열 (가변)
    team_ratio   = pd.crosstab(df['custid'], df['team_nm'],  normalize='index').add_prefix('ratio_team_')
    part_ratio   = pd.crosstab(df['custid'], df['part_nm'],  normalize='index').add_prefix('ratio_part_')
    season_ratio = pd.crosstab(df['custid'], df['season'],   normalize='index').add_prefix('ratio_season_')

    # ⑧ 날짜 확장 (신규 10개)
    def avg_interval(x):
        dates = x.sort_values().dt.normalize().unique()
        if len(dates) < 2:
            return 0.0
        return float(np.diff(dates).astype('timedelta64[D]').astype(int).mean())

    q_ratio = pd.crosstab(df['custid'], df['quarter'], normalize='index')

    date_feats = pd.DataFrame(index=base.index)
    date_feats['date_active_span']         = g['sales_datetime'].apply(lambda x: (x.max() - x.min()).days)
    date_feats['date_avg_interval']        = g['sales_datetime'].apply(avg_interval)
    date_feats['date_recency']             = g['sales_datetime'].apply(lambda x: (max_date - x.max()).days)
    date_feats['date_first_month']         = g['sales_datetime'].apply(lambda x: x.min().month)
    date_feats['date_night_ratio']         = g['is_night_new'].mean()
    date_feats['date_monday_ratio']        = g['is_monday'].mean()
    date_feats['date_active_months']       = g['sales_datetime'].apply(
        lambda x: x.dt.to_period('M').nunique())
    date_feats['date_txn_per_month']       = g['sales_datetime'].apply(
        lambda x: len(x) / max(x.dt.to_period('M').nunique(), 1))
    date_feats['date_early_morning_ratio'] = g['is_early'].mean()
    date_feats['date_max_quarter_ratio']   = q_ratio.max(axis=1)

    # ⑨ 금액 확장 (신규 10개)
    def import_mean_net(sub):
        v = sub[sub['import_flg'] == 1]['net_amt']
        return v.mean() if len(v) > 0 else 0.0
    def domestic_mean_net(sub):
        v = sub[sub['import_flg'] == 0]['net_amt']
        return v.mean() if len(v) > 0 else 0.0

    amt_ext = pd.DataFrame(index=base.index)
    amt_ext['amt_inst_fee_sum']          = g['inst_fee'].sum()
    amt_ext['amt_inst_fee_ratio']        = g['inst_fee'].mean()
    amt_ext['amt_high_price_ratio']      = g['is_high_price'].mean()
    amt_ext['amt_low_price_ratio']       = g['is_low_price'].mean()
    amt_ext['amt_import_mean_net']       = g.apply(import_mean_net)
    amt_ext['amt_domestic_mean_net']     = g.apply(domestic_mean_net)
    amt_ext['amt_import_domestic_ratio'] = (
        amt_ext['amt_import_mean_net'] / (amt_ext['amt_domestic_mean_net'].abs() + 1))
    amt_ext['amt_cv_net']                = g['net_amt'].std() / (g['net_amt'].mean().abs() + 1)
    amt_ext['amt_max_to_total_ratio']    = g['net_amt'].max() / (g['net_amt'].sum().abs() + 1)
    amt_ext['amt_store_net_std']         = g.apply(
        lambda x: x.groupby('str_nm')['net_amt'].sum().std()).fillna(0)

    # ⑩ 카테고리 확장 (신규 10개)
    buyer_ratio = pd.crosstab(df['custid'], df['buyer_nm'], normalize='index')
    brd_ratio   = pd.crosstab(df['custid'], df['brd_nm'],   normalize='index')
    pc_ratio    = pd.crosstab(df['custid'], df['pc_nm'],    normalize='index')

    def corner_switch(x):
        corners = x.sort_values('sales_datetime')['corner_nm'].values
        return int((corners[:-1] != corners[1:]).sum()) if len(corners) > 1 else 0

    cat_feats = pd.DataFrame(index=base.index)
    cat_feats['cat_buyer_diversity']       = g['buyer_nm'].nunique() / g['buyer_nm'].count()
    cat_feats['cat_max_buyer_ratio']       = buyer_ratio.max(axis=1)
    cat_feats['cat_max_brand_ratio']       = brd_ratio.max(axis=1)
    cat_feats['cat_brand_repeat_mean']     = g.apply(lambda x: x['brd_nm'].value_counts().mean())
    cat_feats['cat_corner_switch_cnt']     = g.apply(corner_switch)
    cat_feats['cat_corner_to_brand_ratio'] = g['corner_nm'].nunique() / (g['brd_nm'].nunique() + 1)
    cat_feats['cat_pc_diversity']          = g['pc_nm'].nunique() / g['pc_nm'].count()
    cat_feats['cat_max_pc_ratio']          = pc_ratio.max(axis=1)
    cat_feats['cat_cosmetic_buyer_ratio']  = g['is_cosmetic'].mean()
    cat_feats['cat_luxury_buyer_ratio']    = g['is_luxury'].mean()

    # 전체 결합
    feats = pd.concat([
        base, amt, behavior, ent,
        team_ratio, part_ratio, season_ratio,
        date_feats, amt_ext, cat_feats,
    ], axis=1)

    return feats.fillna(0).reset_index()

features_train = make_features(X_train_trans)
features_test  = make_features(X_test_trans)
print('Manual_Extended features:', features_train.shape[1] - 1)

# ===== train/test feature 컬럼 정렬 (crosstab 류 카테고리 불일치 방지) =====
feat_cols_ref = [c for c in features_train.columns if c != 'custid']
# test 를 train 컬럼에 맞춤: train 에만 있는 건 0 채움, test 에만 있는 건 버림
features_test = features_test.reindex(columns=['custid'] + feat_cols_ref, fill_value=0)
print('정렬 후 feature 수:', len(feat_cols_ref))

Manual_Extended features: 106
정렬 후 feature 수: 106
time: 4min 25s (started: 2026-06-07 17:36:19 +09:00)


## 4. 5-fold 외부 CV (AutoGluon = 하나의 블랙박스 모델)

In [18]:
# ===== 5-fold 외부 CV: AutoGluon 을 하나의 블랙박스 모델로 취급 =====
# - 각 fold: train fold 로 AutoGluon fit -> valid fold 예측 = OOF (leakage 없음)
# - test: 5개 fold 모델 예측의 평균 = test meta feature
SETNAME = SETNAME  # 위에서 정의됨

# custid / target / fold 정렬 맞추기
data = features_train.merge(folds_df, on='custid', how='inner') \
                     .merge(y_train[['custid', 'gender']], on='custid', how='inner')
data = data.sort_values('custid').reset_index(drop=True)

feat_cols = [c for c in features_train.columns if c != 'custid']

oof_pred  = np.zeros(len(data))
test_pred = np.zeros(len(features_test))
test_sorted = features_test.sort_values('custid').reset_index(drop=True)

for f in range(N_SPLITS):
    tr_idx = data.index[data['fold'] != f]
    va_idx = data.index[data['fold'] == f]

    train_part = data.loc[tr_idx, feat_cols + ['gender']]
    valid_part = data.loc[va_idx, feat_cols]

    predictor = TabularPredictor(
        label='gender', problem_type='binary',
        eval_metric=EVAL_METRIC, verbosity=0,
    ).fit(train_data=train_part, presets=PRESETS, time_limit=TIME_LIMIT)

    pos = predictor.positive_class
    oof_pred[va_idx] = predictor.predict_proba(valid_part)[pos].values
    test_pred += predictor.predict_proba(test_sorted[feat_cols])[pos].values / N_SPLITS

    fold_auc = roc_auc_score(data.loc[va_idx, 'gender'], oof_pred[va_idx])
    print(f'[fold {f}] valid AUC = {fold_auc:.5f}')

		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A qui

[fold 0] valid AUC = 0.69654


		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.


[fold 1] valid AUC = 0.69376


		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A qui

[fold 2] valid AUC = 0.70266


		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A qui

[fold 3] valid AUC = 0.68295


		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
		`import lightgbm` failed. A quick tip is to install via `pip install autogluon.tabular[lightgbm]==1.5.0`.
		`import lightgbm` failed. A quick ti

[fold 4] valid AUC = 0.70761
time: 26min 15s (started: 2026-06-07 17:40:44 +09:00)


## 5. CV 지표

In [20]:
# ===== CV 지표: AUC(순위) + LogLoss/Brier(칼리브레이션·오답 페널티) =====
y_true = data['gender'].values
cv_auc   = roc_auc_score(y_true, oof_pred)
cv_ll    = log_loss(y_true, oof_pred)
cv_brier = brier_score_loss(y_true, oof_pred)

print(f'=== {SETNAME} CV ===')
print(f'AUC      : {cv_auc:.5f}   (순위 품질, 높을수록 좋음)')
print(f'LogLoss  : {cv_ll:.5f}   (확신한 오답에 큰 페널티, 낮을수록 좋음)')
print(f'Brier    : {cv_brier:.5f}   (확률 칼리브레이션, 낮을수록 좋음)')

=== manual_extended CV ===
AUC      : 0.69654   (순위 품질, 높을수록 좋음)
LogLoss  : 0.55960   (확신한 오답에 큰 페널티, 낮을수록 좋음)
Brier    : 0.18947   (확률 칼리브레이션, 낮을수록 좋음)
time: 16 ms (started: 2026-06-07 18:07:00 +09:00)


## 6. OOF / test 예측 저장

In [24]:
# ===== OOF / test 예측 저장 (스키마: custid, pred_proba) =====
oof_out = pd.DataFrame({
    'custid': data['custid'].values,
    'pred_proba': oof_pred,
    'fold': data['fold'].values,
})
oof_out.to_csv(f'{OOF_DIR}/f/oof_{SETNAME}_f.csv', index=False)

test_out = pd.DataFrame({
    'custid': test_sorted['custid'].values,
    'pred_proba': test_pred,
})
test_out.to_csv(f'{OOF_DIR}/f/test_{SETNAME}_f.csv', index=False)

print('저장 완료:')
print(f'  {OOF_DIR}/f/oof_{SETNAME}_f.csv      ', oof_out.shape)
print(f'  {OOF_DIR}/f/test_{SETNAME}_f.csv ', test_out.shape)
display(oof_out.head())

저장 완료:
  ../../outputs/oof/1/oof_manual_extended_1.csv       (30000, 3)
  ../../outputs/oof/1/test_manual_extended_1.csv  (19995, 2)


,custid,pred_proba,fold
0,0,0.482732,3
1,1,0.296727,1
2,2,0.612518,0
3,3,0.304221,0
4,4,0.175873,0


time: 63 ms (started: 2026-06-07 18:07:11 +09:00)
